# Method 2 — 01b CE supervision repair
Inputs: complete validation notebook 01 output (includes baseline CE and parent bundle), plus ce_repair_v1 addon. GPU T4 with Internet. No round2 input needed here.
Train fresh XLM-R with corrected mention labels, same 2+2 curriculum. Compare both checkpoints on validation only; do not use test to select. A failed quality gate remains failed even when the run completes.


In [1]:
from pathlib import Path
import os, shutil, subprocess, sys, json
from pathlib import Path
import hashlib, json

def file_digest(path: Path) -> str:
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def choose_bundle(input_root: Path) -> Path:
    candidates = []
    for manifest in sorted(input_root.rglob("completion_manifest.json")):
        data = json.loads(manifest.read_text(encoding="utf-8"))
        if data.get("files") and all((manifest.parent / name).is_file() for name in data["files"]):
            candidates.append(manifest)
    if not candidates or len({file_digest(p) for p in candidates}) != 1:
        raise RuntimeError(f"Attach one complete bundle identity. Complete candidates: {[str(p) for p in candidates]}")
    print("Bundle selected:", candidates[0].parent)
    return candidates[0].parent

def choose_stage(input_root: Path, stage: str, bundle_digest: str) -> Path:
    candidates = []
    identities = set()
    for marker in sorted(input_root.rglob(f"{stage}_completed.json")):
        if marker.parts[-4:-1] != ("results", "method2", "completion"):
            continue
        root = marker.parents[3]
        data = json.loads(marker.read_text(encoding="utf-8"))
        hashes = data.get("checkpoint_hashes", {})
        if data.get("stage") != stage or data.get("completed") is not True or not hashes:
            continue
        if data.get("bundle_manifest_sha256") != bundle_digest:
            print("Skipping marker from another bundle:", marker)
            continue
        invalid = [name for name, digest in hashes.items()
                   if not (root / name).is_file() or file_digest(root / name) != digest]
        if invalid:
            print("Skipping incomplete or mismatched stage output:", marker, invalid[:3])
            continue
        candidates.append(marker)
        identities.add(json.dumps(hashes, sort_keys=True))
    if not candidates or len(identities) != 1:
        raise RuntimeError(f"Attach one complete {stage} checkpoint identity. Verified candidates: {[str(p) for p in candidates]}")
    print(f"{stage} input selected:", candidates[0])
    return candidates[0]


os.environ["CUDA_VISIBLE_DEVICES"] = "0"
bundle = choose_bundle(Path("/kaggle/input"))
work = Path("/kaggle/working")
for name in ["src", "scripts", "configs", "data", "docs", "notebooks"]:
    shutil.copytree(bundle / name, work / name, dirs_exist_ok=True)
shutil.copy2(bundle / "same_domain_feasibility.json", work / "same_domain_feasibility.json")
shutil.copy2(bundle / "completion_manifest.json", work / "completion_manifest.json")
os.chdir(work)
caches = list(Path("/kaggle/input").rglob("models--BAAI--bge-m3"))
if caches:
    cache_hub = caches[0].parent
    os.environ["HF_HUB_CACHE"] = str(cache_hub)
    os.environ["HF_HOME"] = str(cache_hub.parent)
pins = json.loads(Path("configs/method2/pinned_versions.json").read_text())["pinned"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[f"{k}=={v}" for k,v in pins.items()], "jsonschema", "pyyaml", "matplotlib", "rank_bm25", "datasets", "accelerate"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
lora_smoke = """import torch
from transformers import XLMRobertaConfig, XLMRobertaModel
from peft import LoraConfig
config = XLMRobertaConfig(vocab_size=32, hidden_size=8, num_hidden_layers=1, num_attention_heads=2, intermediate_size=16, max_position_embeddings=32)
model = XLMRobertaModel(config)
model.add_adapter(LoraConfig(r=2, lora_alpha=4, target_modules=["query", "value"]))
model(torch.tensor([[0, 5, 2]])).last_hidden_state.square().mean().backward()
assert any(p.grad is not None for n, p in model.named_parameters() if "lora_" in n)
print("LoRA environment smoke passed")
"""
subprocess.run([sys.executable, "-c", lora_smoke], check=True)


Bundle selected: /kaggle/input/datasets/dathq12/output-method2-completion-01-validation/output_method2-completion-01-validation
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.6/739.6 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 33.0 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
LoRA environment smoke passed


CompletedProcess(args=['/usr/bin/python3', '-c', 'import torch\nfrom transformers import XLMRobertaConfig, XLMRobertaModel\nfrom peft import LoraConfig\nconfig = XLMRobertaConfig(vocab_size=32, hidden_size=8, num_hidden_layers=1, num_attention_heads=2, intermediate_size=16, max_position_embeddings=32)\nmodel = XLMRobertaModel(config)\nmodel.add_adapter(LoraConfig(r=2, lora_alpha=4, target_modules=["query", "value"]))\nmodel(torch.tensor([[0, 5, 2]])).last_hidden_state.square().mean().backward()\nassert any(p.grad is not None for n, p in model.named_parameters() if "lora_" in n)\nprint("LoRA environment smoke passed")\n'], returncode=0)

In [2]:
baseline_marker = choose_stage(Path("/kaggle/input"), "validation", file_digest(work / "completion_manifest.json"))
baseline_root = baseline_marker.parents[3]
baseline_meta = json.loads(baseline_marker.read_text())
baseline_path = Path(baseline_meta["results"]["validation_gate"]["checkpoint"])
baseline = work / "artifacts/method2/crossencoder/baseline_for_repair/final"
shutil.copytree(baseline_root / baseline_path, baseline)
patches = list(Path("/kaggle/input").rglob("ce_repair_manifest.json"))
if len(patches) != 1:
    raise RuntimeError(f"Attach exactly one CE repair addon; found {len(patches)}")
patch_path = patches[0]
patch_data = json.loads(patch_path.read_text())
parent_data = json.loads((work / "completion_manifest.json").read_text())
if patch_data["parent_completion_sha256"] != file_digest(work / "completion_manifest.json"):
    raise RuntimeError("CE addon belongs to another bundle")
if set(patch_data["files"]) & set(parent_data["files"]):
    raise RuntimeError("CE addon must not overwrite frozen bundle files")
for name, digest in patch_data["files"].items():
    relative = Path(name)
    if relative.is_absolute() or ".." in relative.parts:
        raise RuntimeError("Invalid addon path")
    source = patch_path.parent / relative
    if file_digest(source) != digest:
        raise RuntimeError(f"Addon hash mismatch: {name}")
    target = work / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)
shutil.copy2(patch_path, work / "ce_repair_manifest.json")
print((work / "ce_repair_audit.json").read_text())


Skipping incomplete or mismatched stage output: /kaggle/input/datasets/dathq12/output-method2-completion-01-validation/output_method2-completion-01-validation/method2_completion_validation_reports/results/method2/completion/validation_completed.json ['artifacts/method2/crossencoder/run01/final/config.json', 'artifacts/method2/crossencoder/run01/final/model.safetensors', 'artifacts/method2/crossencoder/run01/final/tokenizer.json']
validation input selected: /kaggle/input/datasets/dathq12/output-method2-completion-01-validation/output_method2-completion-01-validation/results/method2/completion/validation_completed.json
{
  "parent_completion_sha256": "d562aa4b38bbba20eb28adfc1702df42c193add4795c7415528a62bde945c908",
  "custom_train_sha256": "98f55af4e173894b9a97e93ee6475e7070e570e730f0658f28effea8aa411ae6",
  "splits": {
    "train": {
      "old_custom_pairs": 12253,
      "new_custom_pairs": 15318,
      "added_pairs": 3065,
      "corrected_span_boundaries": 693,
      "retained_samp

In [3]:
subprocess.run([sys.executable, "scripts/method2/run_ce_repair.py"], check=True)
print(Path("results/method2/ce_repair/comparison.json").read_text())


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2259.19it/s]


{
  "passed": false,
  "failures": [
    {
      "metric": "span_em",
      "value": 0.7564,
      "required": 0.8
    },
    {
      "metric": "enum_accuracy",
      "value": 0.7571,
      "required": 0.9
    },
    {
      "metric": "argument_em",
      "value": 0.45652173913043476,
      "required": 0.7
    }
  ],
  "missing_metrics": [],
  "complete": true,
  "measured_on": "custom_val_seen+custom_val_unseen",
  "inference_diagnostics": {
    "argument_em": 0.45652173913043476,
    "gold_call_count": 460,
    "correct_call_count": 210,
    "values_on_gold": {
      "string": {
        "correct": 598,
        "support": 680,
        "accuracy": 0.8794117647058823
      },
      "required": {
        "correct": 764,
        "support": 947,
        "accuracy": 0.8067581837381204
      },
      "integer": {
        "correct": 185,
        "support": 283,
        "accuracy": 0.6537102473498233
      },
      "boolean": {
        "correct": 90,
        "support": 91,
        "accuracy": 

[crossencoder] HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/config.json "HTTP/1.1 200 OK"
[crossencoder] HTTP Request: GET https://huggingface.co/xlm-roberta-base/resolve/main/config.json "HTTP/1.1 200 OK"
[crossencoder] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
[crossencoder] HTTP Request: HEAD https://huggingface.co/xlm-roberta-base/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
[crossencoder] HTTP Request: GET https://huggingface.co/xlm-roberta-base/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
[crossencoder] HTTP Request: GET https://huggingface.co/api/models/xlm-roberta-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
[crossencoder] HTTP Request: GET https://huggingface.co/api/models/FacebookAI/xlm-roberta-base/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not 

{
  "has_value_precision": 0.9668,
  "has_value_recall": 0.9775,
  "has_value_f1": 0.9721,
  "has_value_accuracy": 0.9548,
  "span_em": 0.7526,
  "span_start_accuracy": 0.8248,
  "enum_accuracy": 0.7232,
  "boolean_accuracy": 0.9444,
  "n_span": 13935,
  "n_enum": 271,
  "n_boolean": 324,
  "loss": 1.5816
}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2151.13it/s]


{
  "passed": false,
  "failures": [
    {
      "metric": "enum_accuracy",
      "value": 0.7449,
      "required": 0.9
    },
    {
      "metric": "argument_em",
      "value": 0.6195652173913043,
      "required": 0.7
    }
  ],
  "missing_metrics": [],
  "complete": true,
  "measured_on": "custom_val_seen+custom_val_unseen",
  "inference_diagnostics": {
    "argument_em": 0.6195652173913043,
    "gold_call_count": 460,
    "correct_call_count": 285,
    "values_on_gold": {
      "string": {
        "correct": 600,
        "support": 680,
        "accuracy": 0.8823529411764706
      },
      "required": {
        "correct": 822,
        "support": 947,
        "accuracy": 0.8680042238648363
      },
      "integer": {
        "correct": 283,
        "support": 283,
        "accuracy": 1.0
      },
      "boolean": {
        "correct": 90,
        "support": 91,
        "accuracy": 0.989010989010989
      },
      "number": {
        "correct": 54,
        "support": 59,
        "ac

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2141.17it/s]


{
  "passed": false,
  "failures": [
    {
      "metric": "enum_accuracy",
      "value": 0.7449,
      "required": 0.9
    },
    {
      "metric": "argument_em",
      "value": 0.6195652173913043,
      "required": 0.7
    }
  ],
  "missing_metrics": [],
  "complete": true,
  "measured_on": "custom_val_seen+custom_val_unseen",
  "inference_diagnostics": {
    "argument_em": 0.6195652173913043,
    "gold_call_count": 460,
    "correct_call_count": 285,
    "values_on_gold": {
      "string": {
        "correct": 600,
        "support": 680,
        "accuracy": 0.8823529411764706
      },
      "required": {
        "correct": 822,
        "support": 947,
        "accuracy": 0.8680042238648363
      },
      "integer": {
        "correct": 283,
        "support": 283,
        "accuracy": 1.0
      },
      "boolean": {
        "correct": 90,
        "support": 91,
        "accuracy": 0.989010989010989
      },
      "number": {
        "correct": 54,
        "support": 59,
        "ac

In [4]:
archive = work / "method2_ce_repair_reports.tar.gz"
subprocess.run(["tar", "czf", str(archive), "results/method2/ce_repair", "results/method2/completion", "ce_repair_audit.json", "ce_repair_manifest.json"], check=True)
print("Save the whole notebook output. The reports archive omits model weights.")
print(archive)


Save the whole notebook output. The reports archive omits model weights.
/kaggle/working/method2_ce_repair_reports.tar.gz
